# Notebook 02 — Amino Acids and Protein Primary Structure

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 02 of 10  
**Prerequisites:** Module 05 NB01–NB03 (central dogma, proteins)  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Every protein is a sequence of amino acids. The sequence determines the structure,
the structure determines the function, and the function is what we care about in
biology. Before we can reason about protein function — or read a proteomics paper,
or understand why AlphaFold works — we need the vocabulary: the 20 amino acids,
their properties, and how they combine into a primary sequence.

---
## Step 3 — Biological Background

### The 20 standard amino acids — grouped by R-group property

| Group | Amino acids (1-letter) | Key property |
|-------|----------------------|--------------|
| Nonpolar / hydrophobic | G, A, V, L, I, P, F, W, M | Interior of folded proteins; drive hydrophobic collapse |
| Polar, uncharged | S, T, C, Y, N, Q | Hydrogen bonds; surface-accessible |
| Positively charged (basic) | K, R, H | Positive at physiological pH (pKa > 7); interact with DNA |
| Negatively charged (acidic) | D, E | Negative at physiological pH; metal ion binding |
| Special | G (no R), P (cyclic — restricts backbone flexibility), C (disulfide bonds) | |

### Peptide bond formation

Two amino acids join via a condensation reaction: the α-carboxyl group of one
reacts with the α-amino group of the next, releasing water. The resulting **peptide
bond** is planar (partial double bond character) — the key structural constraint
that limits backbone conformations (Ramachandran plot).

### Primary structure → higher structures

- **Primary:** the amino acid sequence
- **Secondary:** local patterns: α-helix (H-bonds every 4 residues), β-strand/sheet
- **Tertiary:** overall 3D fold of one polypeptide
- **Quaternary:** multiple polypeptide subunits (e.g. haemoglobin: α₂β₂)

### Molecular weight

Average monoisotopic residue masses (Da) vary by amino acid. Average residue mass
≈ 110 Da. A 400-residue protein ≈ 44 kDa.

---
## Step 4 — Mathematical Explanation

**Molecular weight of a protein from sequence:**
$$M_W = \sum_{i=1}^{N} m_{aa_i} - (N-1) \times 18.015$$
where m_{aa_i} is the molecular weight of each amino acid (as free acid) and
we subtract 18.015 Da (water) for each peptide bond formed (N-1 bonds for N residues).

**Isoelectric point (pI):** the pH at which net charge = 0. For a simple dipeptide,
pI ≈ (pKa1 + pKa2) / 2. For a full protein, requires iterative numerical solution.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Cell 6.1 — Amino acid properties database
AA_PROPERTIES = {
    # one-letter: (name, group, mw_free_acid, hydrophobicity_kyte_doolittle)
    'A': ('Alanine',       'nonpolar',   89.1,  1.8),
    'R': ('Arginine',      'positive',  174.2, -4.5),
    'N': ('Asparagine',    'polar',     132.1, -3.5),
    'D': ('Aspartate',     'negative',  133.1, -3.5),
    'C': ('Cysteine',      'polar',     121.2,  2.5),
    'Q': ('Glutamine',     'polar',     146.2, -3.5),
    'E': ('Glutamate',     'negative',  147.1, -3.5),
    'G': ('Glycine',       'nonpolar',   75.0, -0.4),
    'H': ('Histidine',     'positive',  155.2, -3.2),
    'I': ('Isoleucine',    'nonpolar',  131.2,  4.5),
    'L': ('Leucine',       'nonpolar',  131.2,  3.8),
    'K': ('Lysine',        'positive',  146.2, -3.9),
    'M': ('Methionine',    'nonpolar',  149.2,  1.9),
    'F': ('Phenylalanine', 'nonpolar',  165.2,  2.8),
    'P': ('Proline',       'special',   115.1, -1.6),
    'S': ('Serine',        'polar',     105.1, -0.8),
    'T': ('Threonine',     'polar',     119.1, -0.7),
    'W': ('Tryptophan',    'nonpolar',  204.2, -0.9),
    'Y': ('Tyrosine',      'polar',     181.2, -1.3),
    'V': ('Valine',        'nonpolar',  117.1,  4.2),
}

GROUP_COLORS = {'nonpolar': '#4ECDC4', 'polar': '#95E1D3',
                'positive': '#6C63FF', 'negative': '#F38181', 'special': '#FFD166'}

print("Amino acids by group:")
for group in ['nonpolar', 'polar', 'positive', 'negative', 'special']:
    aas = [aa for aa, (_, g, _, _) in AA_PROPERTIES.items() if g == group]
    print(f"  {group:>12}: {', '.join(aas)}")

In [ ]:
# Cell 6.2 — Molecular weight from amino acid sequence
WATER_MW = 18.015

def molecular_weight(seq: str) -> float:
    """
    Calculate molecular weight of a protein from its one-letter sequence.
    Uses free-acid masses and subtracts water for each peptide bond.
    """
    seq = seq.upper()
    mw = sum(AA_PROPERTIES[aa][2] for aa in seq if aa in AA_PROPERTIES)
    n_bonds = sum(1 for aa in seq if aa in AA_PROPERTIES) - 1
    return mw - n_bonds * WATER_MW


def amino_acid_composition(seq: str) -> dict:
    """Count amino acid frequencies and classify by group."""
    from collections import Counter
    seq = seq.upper()
    counts = Counter(aa for aa in seq if aa in AA_PROPERTIES)
    total = sum(counts.values())
    return {
        'counts': dict(counts),
        'fractions': {aa: n/total for aa, n in counts.items()},
        'group_fractions': {
            group: sum(counts.get(aa, 0) for aa, (_, g, _, _) in AA_PROPERTIES.items()
                       if g == group) / total
            for group in GROUP_COLORS
        }
    }


def hydrophobicity_profile(seq: str, window: int = 9) -> np.ndarray:
    """
    Kyte-Doolittle hydrophobicity profile with sliding window average.
    Positive values → hydrophobic (likely membrane-spanning or buried)
    """
    seq = seq.upper()
    scores = np.array([AA_PROPERTIES.get(aa, ('', '', 0, 0))[3] for aa in seq])
    half = window // 2
    profile = np.array([
        scores[max(0, i-half):min(len(scores), i+half+1)].mean()
        for i in range(len(scores))
    ])
    return profile


# Test: human insulin B-chain
insulin_B = 'FVNQHLCGSHLVEALYLVCGERGFFYTPKT'
mw = molecular_weight(insulin_B)
comp = amino_acid_composition(insulin_B)

print(f"Insulin B-chain ({len(insulin_B)} residues):")
print(f"  MW = {mw:.1f} Da ({mw/1000:.2f} kDa)")
print(f"  Expected: ~3.5 kDa")
print(f"  Composition by group:")
for group, frac in comp['group_fractions'].items():
    print(f"    {group:>12}: {100*frac:.1f}%")

In [ ]:
# Cell 6.3 — Net charge as a function of pH
# pKa values: alpha-NH3+ (N-term), alpha-COOH (C-term), and R-groups
PKA_NTERM = 8.0
PKA_CTERM = 3.1
PKA_RGROUP = {
    'D': 3.9, 'E': 4.1, 'H': 6.0, 'C': 8.3, 'Y': 10.1, 'K': 10.5, 'R': 12.5
}

def net_charge(seq: str, pH: float) -> float:
    """
    Compute approximate net charge of a protein at a given pH.
    Uses Henderson-Hasselbalch: fraction ionised from pKa.
    """
    seq = seq.upper()
    charge = 0.0

    # N-terminus (positively charged when protonated)
    charge += 1 / (1 + 10**(pH - PKA_NTERM))

    # C-terminus (negatively charged when deprotonated)
    charge -= 1 / (1 + 10**(PKA_CTERM - pH))

    # R-groups
    for aa in seq:
        if aa in ('D', 'E'):  # acidic: negative when deprotonated
            charge -= 1 / (1 + 10**(PKA_RGROUP[aa] - pH))
        elif aa in ('H', 'K', 'R'):  # basic: positive when protonated
            charge += 1 / (1 + 10**(pH - PKA_RGROUP[aa]))
        elif aa == 'C':  # thiol: negative when deprotonated
            charge -= 1 / (1 + 10**(PKA_RGROUP['C'] - pH))
        elif aa == 'Y':  # phenol: negative when deprotonated
            charge -= 1 / (1 + 10**(PKA_RGROUP['Y'] - pH))

    return charge

# Find pI (where net charge = 0)
ph_range = np.linspace(0, 14, 1000)
charges = [net_charge(insulin_B, ph) for ph in ph_range]
pI_idx = np.argmin(np.abs(charges))
print(f"Insulin B-chain pI ≈ {ph_range[pI_idx]:.2f}  (expected: ~5.4)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Amino acid hydrophobicity chart + hydrophobicity profile
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: Amino acid hydrophobicity (Kyte-Doolittle) coloured by group
ax = axes[0]
aas_sorted = sorted(AA_PROPERTIES.keys(),
                     key=lambda a: AA_PROPERTIES[a][3])
hydros = [AA_PROPERTIES[a][3] for a in aas_sorted]
groups = [AA_PROPERTIES[a][1] for a in aas_sorted]
colors = [GROUP_COLORS[g] for g in groups]

bars = ax.barh(aas_sorted, hydros, color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Kyte-Doolittle hydrophobicity')
ax.set_title('Amino acid hydrophobicity\n(positive = hydrophobic, negative = hydrophilic)')
handles = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=handles, fontsize=7, loc='lower right')

# Panel 2: Hydrophobicity profile of insulin B-chain
ax = axes[1]
profile = hydrophobicity_profile(insulin_B, window=7)
x = np.arange(len(profile))
ax.fill_between(x, profile, 0, where=(profile > 0), color='tomato', alpha=0.6, label='Hydrophobic')
ax.fill_between(x, profile, 0, where=(profile <= 0), color='steelblue', alpha=0.6, label='Hydrophilic')
ax.plot(x, profile, 'k-', lw=1)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(insulin_B)))
ax.set_xticklabels(list(insulin_B), fontsize=7)
ax.set_xlabel('Position')
ax.set_ylabel('Hydrophobicity (7-aa window)')
ax.set_title('Insulin B-chain hydrophobicity profile')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `molecular_weight(seq)` from scratch. Verify: insulin B-chain (30 residues)
   should give ≈ 3430 Da. Compare your result to ExPASy ProtParam.
2. A protein with sequence 'MAAADKKHHHHHH' has a His-tag (6× His). At pH 7.4,
   what is the approximate net charge? Why is this useful for Ni-NTA purification?
3. Implement `hydrophobicity_profile(seq, window)`. Which residues in insulin B-chain
   are most hydrophobic? How does window size affect the profile?
4. Proline is described as a 'helix breaker'. Why? (Hint: think about the cyclic
   structure and the φ backbone dihedral angle constraint.)

---
## Quiz — Active Recall

1. Name one amino acid from each of the four R-group classes.
2. What is the peptide bond? Why is it planar?
3. At pH 7, is lysine (K) positively or negatively charged? Why?
4. What does a Kyte-Doolittle score > 1.6 suggest about a protein segment?
5. What is the molecular weight rule of thumb for a typical protein domain?

---
## Reflection

**Date completed:** ____________________

> *[If a GWAS identified a missense variant K57E (Lysine → Glutamate), what type of
> R-group change is this? What effect might it have on a protein that interacts with
> DNA?]*

---
*Next: `03_enzyme_kinetics_michaelis_menten.ipynb`*